# Copy Delay Simulation

Question: if I copied this trader after delay X, would the apparent edge survive?

This notebook reports slippage, missing-price ratio, and worse-than-guru ratio.

In [1]:
import pandas as pd

from polymarket_insight.analysis.copy_delay import simulate_copy_delay
from polymarket_insight.data.datasets import traders
from polymarket_insight.data.storage.normalized import NormalizedStore

wallet = "0x0000000000000000000000000000000000000000"
store = NormalizedStore()
trader_trades = traders.trades(wallet, store=store)
price_points = store.read_table("price_points")
copy_delay = simulate_copy_delay(
    trader_trades,
    price_points,
    [pd.Timedelta("1m"), pd.Timedelta("5m"), pd.Timedelta("30m"), pd.Timedelta("2h")],
)
copy_delay.head()

,tx_hash,trader_wallet,condition_id,token_id,side,guru_trade_ts,guru_price,delay,copy_ts,copy_price,price_delta,worse_than_guru,missing_price,price_evidence


In [ ]:
if copy_delay.empty:
    summary = {"trade_count_after_filters": 0, "missing_price_ratio": 1.0}
else:
    summary = (
        copy_delay.groupby("delay")
        .agg(
            average_slippage=("price_delta", "mean"),
            median_slippage=("price_delta", "median"),
            worse_than_guru_ratio=("worse_than_guru", "mean"),
            missing_price_ratio=("missing_price", "mean"),
            trade_count_after_filters=("tx_hash", "count"),
        )
        .reset_index()
    )
summary